In [1]:
from __future__ import annotations

from pathlib import Path
import sys


def find_project_root(start_path: Path | None = None) -> Path:
    """Locate the Geospatial-CANOE repository root.

    The search begins from ``start_path`` when supplied, or from the active
    Jupyter working directory otherwise. The first directory containing both
    ``scripts/`` and ``data_files/`` is treated as the repository root.

    Parameters
    ----------
    start_path : Path | None, optional
        Directory from which to begin the upward search. When omitted, the
        current working directory is used.

    Returns
    -------
    Path
        Resolved path to the Geospatial-CANOE repository root.

    Raises
    ------
    FileNotFoundError
        If no searched directory contains both ``scripts/`` and
        ``data_files/``.
    """

    start = (
        Path.cwd().resolve()
        if start_path is None
        else start_path.resolve()
    )

    if start.is_file():
        start = start.parent

    for candidate in (start, *start.parents):
        if (
            (candidate / "scripts").is_dir()
            and (candidate / "data_files").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the Geospatial-CANOE repository root. "
        "Expected to find both scripts/ and data_files/."
    )


PROJECT_ROOT = find_project_root()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
DATA_FILES = PROJECT_ROOT / "data_files"
CONFIG_DIR = PROJECT_ROOT / "config" 

for import_path in (PROJECT_ROOT, SCRIPTS_DIR):
    import_path_string = str(import_path)

    if import_path_string not in sys.path:
        sys.path.insert(0, import_path_string)


print("Geospatial-CANOE notebook environment initialized.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Scripts:      {SCRIPTS_DIR}")
print(f"Data files:   {DATA_FILES}")
print(f"Config files: {CONFIG_DIR}")

Geospatial-CANOE notebook environment initialized.
Project root: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace
Scripts:      C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\scripts
Data files:   C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files
Config files: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\config


In [2]:
from project_config import (
    GeospatialBuildConfig,
    load_geospatial_build_config,
    print_build_config,
)


CONFIG_PATH = CONFIG_DIR / "provinces_only.toml"

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        f"Build-profile TOML file not found: {CONFIG_PATH}"
    )

build_config: GeospatialBuildConfig = (
    load_geospatial_build_config(CONFIG_PATH)
)

print_build_config(build_config)


Geospatial-CANOE build profile
Source:                  C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\config\provinces_only.toml
Study area:              provinces_only
Provinces/territories:  NL, PE, NS, NB, QC, ON, MB, SK, AB, BC
Grid types:             geographic, projected
Geographic resolutions: 0.1°, 0.25°, 0.5°, 0.75°, 1°
Projected resolutions:  10 km, 25 km, 50 km, 75 km, 100 km
Road networks:          backbone, freight_access
Connectivity methods:   weak, strong
Schema road method:     strong
Point boundary buffer:  50 km
Buffer simplification:  5 km
Max snap factor:        2 × grid resolution


In [4]:
import importlib


STAGE_MODULES = {
    "legacy_inputs": "map_legacy_inputs",
    "basemaps": "build_basemaps",
    "adjacency": "build_region_adjacency",
    "roads": "build_roads",
    "road_connectivity": "map_roads",
    "emissions": "build_emissions",
    "h2_pipeline_costs": "build_h2_pipeline_costs",
    "h2_pipeline_cost_models": "build_h2_pipeline_cost_models",
}


stage_modules: dict[str, object] = {}
module_import_results: list[dict[str, str]] = []

for stage_name, module_name in STAGE_MODULES.items():
    try:
        module = importlib.import_module(module_name)
        stage_modules[stage_name] = module

        module_import_results.append(
            {
                "stage": stage_name,
                "module": module_name,
                "status": "imported",
            }
        )

    except Exception as exc:
        module_import_results.append(
            {
                "stage": stage_name,
                "module": module_name,
                "status": (
                    f"failed: {type(exc).__name__}: {exc}"
                ),
            }
        )


print("Silver-stage module imports")
print("=" * 78)

for result in module_import_results:
    print(
        f"{result['stage']:<25}"
        f"{result['status']:<45}"
        f"{result['module']}"
    )


import_failures = [
    result
    for result in module_import_results
    if result["status"] != "imported"
]

print("\nImport summary")
print("=" * 78)
print(f"Configured modules: {len(STAGE_MODULES):,}")
print(f"Imported modules:   {len(stage_modules):,}")
print(f"Import failures:    {len(import_failures):,}")

if import_failures:
    failure_details = "\n".join(
        (
            f"  - {result['stage']} "
            f"({result['module']}): "
            f"{result['status']}"
        )
        for result in import_failures
    )

    raise ImportError(
        "One or more silver-stage modules could not be imported:\n"
        f"{failure_details}"
    )

Silver-stage module imports
legacy_inputs            imported                                     map_legacy_inputs
basemaps                 imported                                     build_basemaps
adjacency                imported                                     build_region_adjacency
roads                    imported                                     build_roads
road_connectivity        imported                                     map_roads
emissions                imported                                     build_emissions
h2_pipeline_costs        imported                                     build_h2_pipeline_costs
h2_pipeline_cost_models  imported                                     build_h2_pipeline_cost_models

Import summary
Configured modules: 8
Imported modules:   8
Import failures:    0


In [5]:
from pathlib import Path


SILVER_STAGE_ORDER = [
    "legacy_inputs",
    "emissions",
    "h2_pipeline_costs",
    "h2_pipeline_cost_models",
    "basemaps",
    "adjacency",
    "roads",
    "road_connectivity",
]


# Dependencies on other stages within the silver build workflow.
SILVER_STAGE_DEPENDENCIES = {
    "legacy_inputs": (),
    "emissions": (),
    "h2_pipeline_costs": (),
    "h2_pipeline_cost_models": (
        "h2_pipeline_costs",
    ),
    "basemaps": (),
    "adjacency": (
        "basemaps",
    ),
    "roads": (),
    "road_connectivity": (
        "basemaps",
        "adjacency",
        "roads",
    ),
}


# Inputs produced outside the silver build workflow.
EXTERNAL_INPUT_DEPENDENCIES = {
    "legacy_inputs": (
        "raw provincial boundary shapefile",
        "data_files/sites_full.csv",
        "data_files/demand.csv",
    ),
    "emissions": (
        "raw emissions CSV",
        "raw emissions GeoJSON",
    ),
    "h2_pipeline_costs": (
        "H2 pipeline master cost workbook",
    ),
    "h2_pipeline_cost_models": (),
    "basemaps": (
        "raw provincial boundary shapefile",
    ),
    "adjacency": (),
    "roads": (
        "raw provincial and territorial NRN GeoPackages",
    ),
    "road_connectivity": (),
}


# Defines how each stage is executed.
#
# Function stages execute an imported run_* function in the current notebook
# process. Script stages execute the existing standalone Python script.
STAGE_EXECUTORS = {
    "legacy_inputs": {
        "method": "function",
        "callable": getattr(
            stage_modules["legacy_inputs"],
            "run_legacy_input_mapping",
        ),
    },
    "emissions": {
        "method": "script",
        "path": SCRIPTS_DIR / "build_emissions.py",
    },
    "h2_pipeline_costs": {
        "method": "script",
        "path": SCRIPTS_DIR / "build_h2_pipeline_costs.py",
    },
    "h2_pipeline_cost_models": {
        "method": "script",
        "path": SCRIPTS_DIR / "build_h2_pipeline_cost_models.py",
    },
    "basemaps": {
        "method": "function",
        "callable": getattr(
            stage_modules["basemaps"],
            "run_basemap_build",
        ),
    },
    "adjacency": {
        "method": "function",
        "callable": getattr(
            stage_modules["adjacency"],
            "run_adjacency_build",
        ),
    },
    "roads": {
        "method": "function",
        "callable": getattr(
            stage_modules["roads"],
            "run_road_build",
        ),
    },
    "road_connectivity": {
        "method": "function",
        "callable": getattr(
            stage_modules["road_connectivity"],
            "run_road_connectivity_build",
        ),
    },
}


# =============================================================================
# Workflow-definition validation
# =============================================================================

unknown_ordered_stages = [
    stage_name
    for stage_name in SILVER_STAGE_ORDER
    if stage_name not in STAGE_MODULES
]

missing_stage_dependency_entries = [
    stage_name
    for stage_name in SILVER_STAGE_ORDER
    if stage_name not in SILVER_STAGE_DEPENDENCIES
]

missing_external_dependency_entries = [
    stage_name
    for stage_name in SILVER_STAGE_ORDER
    if stage_name not in EXTERNAL_INPUT_DEPENDENCIES
]

unknown_dependency_stages = sorted(
    {
        dependency
        for dependencies in SILVER_STAGE_DEPENDENCIES.values()
        for dependency in dependencies
        if dependency not in STAGE_MODULES
    }
)

unordered_registered_stages = [
    stage_name
    for stage_name in STAGE_MODULES
    if stage_name not in SILVER_STAGE_ORDER
]

missing_executor_entries = [
    stage_name
    for stage_name in SILVER_STAGE_ORDER
    if stage_name not in STAGE_EXECUTORS
]

unknown_executor_stages = [
    stage_name
    for stage_name in STAGE_EXECUTORS
    if stage_name not in SILVER_STAGE_ORDER
]

invalid_executor_methods = [
    stage_name
    for stage_name, executor in STAGE_EXECUTORS.items()
    if executor.get("method") not in {"function", "script"}
]

missing_function_callables = [
    stage_name
    for stage_name, executor in STAGE_EXECUTORS.items()
    if (
        executor.get("method") == "function"
        and not callable(executor.get("callable"))
    )
]

missing_script_paths = [
    stage_name
    for stage_name, executor in STAGE_EXECUTORS.items()
    if (
        executor.get("method") == "script"
        and not isinstance(executor.get("path"), Path)
    )
]

nonexistent_script_paths = [
    stage_name
    for stage_name, executor in STAGE_EXECUTORS.items()
    if (
        executor.get("method") == "script"
        and isinstance(executor.get("path"), Path)
        and not executor["path"].is_file()
    )
]


stage_positions = {
    stage_name: index
    for index, stage_name in enumerate(SILVER_STAGE_ORDER)
}

invalid_dependency_order = []

for stage_name, dependencies in SILVER_STAGE_DEPENDENCIES.items():
    for dependency in dependencies:
        if (
            stage_name in stage_positions
            and dependency in stage_positions
            and stage_positions[dependency] >= stage_positions[stage_name]
        ):
            invalid_dependency_order.append(
                (stage_name, dependency)
            )


if unknown_ordered_stages:
    raise ValueError(
        "The silver-stage order contains unregistered stages: "
        f"{unknown_ordered_stages}"
    )

if missing_stage_dependency_entries:
    raise ValueError(
        "The following stages have no silver-stage dependency entry: "
        f"{missing_stage_dependency_entries}"
    )

if missing_external_dependency_entries:
    raise ValueError(
        "The following stages have no external-input dependency entry: "
        f"{missing_external_dependency_entries}"
    )

if unknown_dependency_stages:
    raise ValueError(
        "The dependency graph references unregistered stages: "
        f"{unknown_dependency_stages}"
    )

if unordered_registered_stages:
    raise ValueError(
        "The following registered stages are absent from the workflow order: "
        f"{unordered_registered_stages}"
    )

if missing_executor_entries:
    raise ValueError(
        "The following stages have no executor entry: "
        f"{missing_executor_entries}"
    )

if unknown_executor_stages:
    raise ValueError(
        "The executor registry contains stages absent from the workflow order: "
        f"{unknown_executor_stages}"
    )

if invalid_executor_methods:
    raise ValueError(
        "The following stages use unsupported execution methods: "
        f"{invalid_executor_methods}"
    )

if missing_function_callables:
    raise TypeError(
        "The following function executors do not contain callable objects: "
        f"{missing_function_callables}"
    )

if missing_script_paths:
    raise TypeError(
        "The following script executors do not contain Path objects: "
        f"{missing_script_paths}"
    )

if nonexistent_script_paths:
    raise FileNotFoundError(
        "The following script executors point to missing files: "
        f"{nonexistent_script_paths}"
    )

if invalid_dependency_order:
    formatted_dependencies = ", ".join(
        f"{stage} depends on {dependency}"
        for stage, dependency in invalid_dependency_order
    )

    raise ValueError(
        "One or more dependencies appear after their dependent stage: "
        f"{formatted_dependencies}"
    )


# =============================================================================
# Workflow summary
# =============================================================================

print("Silver workflow definition")
print("=" * 78)

for index, stage_name in enumerate(
    SILVER_STAGE_ORDER,
    start=1,
):
    stage_dependencies = SILVER_STAGE_DEPENDENCIES[stage_name]
    external_dependencies = EXTERNAL_INPUT_DEPENDENCIES[stage_name]
    executor = STAGE_EXECUTORS[stage_name]

    stage_dependency_text = (
        ", ".join(stage_dependencies)
        if stage_dependencies
        else "none"
    )

    external_dependency_text = (
        ", ".join(external_dependencies)
        if external_dependencies
        else "none"
    )

    execution_method = str(executor["method"])

    if execution_method == "function":
        execution_target = executor["callable"].__name__
    else:
        execution_target = str(executor["path"])

    print(f"\n{index:>2}. {stage_name}")
    print(f"    Execution method:    {execution_method}")
    print(f"    Execution target:    {execution_target}")
    print(f"    Silver dependencies: {stage_dependency_text}")
    print(f"    External inputs:     {external_dependency_text}")


function_stage_count = sum(
    executor["method"] == "function"
    for executor in STAGE_EXECUTORS.values()
)

script_stage_count = sum(
    executor["method"] == "script"
    for executor in STAGE_EXECUTORS.values()
)


print("\nWorkflow validation complete.")
print(f"Registered stages: {len(SILVER_STAGE_ORDER):,}")
print(f"Runnable stages:   {len(STAGE_EXECUTORS):,}")
print(f"Function stages:   {function_stage_count:,}")
print(f"Script stages:     {script_stage_count:,}")

Silver workflow definition

 1. legacy_inputs
    Execution method:    function
    Execution target:    run_legacy_input_mapping
    Silver dependencies: none
    External inputs:     raw provincial boundary shapefile, data_files/sites_full.csv, data_files/demand.csv

 2. emissions
    Execution method:    script
    Execution target:    C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\scripts\build_emissions.py
    Silver dependencies: none
    External inputs:     raw emissions CSV, raw emissions GeoJSON

 3. h2_pipeline_costs
    Execution method:    script
    Execution target:    C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\scripts\build_h2_pipeline_costs.py
    Silver dependencies: none
    External inputs:     H2 pipeline master cost workbook

 4. h2_pipeline_cost_models
    Execution method:    script
    Execution target:    C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\scripts\build_h2_pipeline_cost_models.py
  

In [6]:
# =============================================================================
# External input path resolution
# =============================================================================

raw_boundary_dir = DATA_FILES / "raw" / "basemaps"
raw_boundary_paths = sorted(raw_boundary_dir.glob("*.shp"))

sites_path = DATA_FILES / "sites_full.csv"
demand_path = DATA_FILES / "demand.csv"

raw_nrn_dir = DATA_FILES / "raw" / "nrn"

emissions_module = stage_modules["emissions"]
h2_pipeline_cost_module = stage_modules["h2_pipeline_costs"]

raw_emissions_csv_path = emissions_module.CO2_CSV_PATH
raw_emissions_geojson_path = emissions_module.CO2_JSON_PATH

h2_pipeline_workbook_path = (
    h2_pipeline_cost_module.default_workbook_path(
        PROJECT_ROOT
    )
)


# =============================================================================
# External input validation helpers
# =============================================================================

external_input_checks: list[dict[str, object]] = []


def add_external_input_check(
    stage_name: str,
    input_name: str,
    path: Path,
    status: str,
    details: str | None = None,
) -> None:
    """Add one external-input validation result.

    Parameters
    ----------
    stage_name : str
        Silver stage that consumes the external input.
    input_name : str
        Human-readable name of the external input.
    path : Path
        Expected file or directory path.
    status : str
        Validation result, such as ``"found"``, ``"missing"``, or
        ``"invalid"``.
    details : str | None, optional
        Additional validation information.
    """

    external_input_checks.append(
        {
            "stage": stage_name,
            "input": input_name,
            "path": path,
            "status": status,
            "details": details,
        }
    )


def check_required_file(
    stage_name: str,
    input_name: str,
    path: Path,
) -> None:
    """Validate and record one required external file."""

    add_external_input_check(
        stage_name=stage_name,
        input_name=input_name,
        path=path,
        status="found" if path.is_file() else "missing",
    )


# =============================================================================
# Shared raw boundary validation
# =============================================================================

if len(raw_boundary_paths) == 1:
    raw_boundary_path = raw_boundary_paths[0]
    boundary_status = "found"
    boundary_details = None
else:
    raw_boundary_path = raw_boundary_dir
    boundary_status = "invalid"
    boundary_details = (
        "Expected exactly one boundary shapefile, found "
        f"{len(raw_boundary_paths)}: "
        f"{[path.name for path in raw_boundary_paths]}"
    )

for dependent_stage in (
    "legacy_inputs",
    "basemaps",
):
    add_external_input_check(
        stage_name=dependent_stage,
        input_name="raw provincial boundary shapefile",
        path=raw_boundary_path,
        status=boundary_status,
        details=boundary_details,
    )


# =============================================================================
# Legacy input validation
# =============================================================================

check_required_file(
    stage_name="legacy_inputs",
    input_name="legacy site table",
    path=sites_path,
)

check_required_file(
    stage_name="legacy_inputs",
    input_name="legacy demand table",
    path=demand_path,
)


# =============================================================================
# Emissions input validation
# =============================================================================

check_required_file(
    stage_name="emissions",
    input_name="raw emissions CSV",
    path=raw_emissions_csv_path,
)

check_required_file(
    stage_name="emissions",
    input_name="raw emissions GeoJSON",
    path=raw_emissions_geojson_path,
)


# =============================================================================
# H2 pipeline cost input validation
# =============================================================================

check_required_file(
    stage_name="h2_pipeline_costs",
    input_name="H2 pipeline master cost workbook",
    path=h2_pipeline_workbook_path,
)


# =============================================================================
# National Road Network input validation
# =============================================================================

for province in build_config.study_area.provinces:
    province_directory = raw_nrn_dir / province

    nrn_matches = (
        sorted(province_directory.glob("*_en.gpkg"))
        if province_directory.is_dir()
        else []
    )

    if len(nrn_matches) == 1:
        nrn_path = nrn_matches[0]
        nrn_status = "found"
        nrn_details = None

    elif not province_directory.is_dir():
        nrn_path = province_directory
        nrn_status = "missing"
        nrn_details = (
            "Province or territory NRN directory does not exist."
        )

    else:
        nrn_path = province_directory
        nrn_status = "invalid"
        nrn_details = (
            "Expected exactly one English NRN GeoPackage matching "
            f"'*_en.gpkg', found {len(nrn_matches)}: "
            f"{[path.name for path in nrn_matches]}"
        )

    add_external_input_check(
        stage_name="roads",
        input_name=f"{province} English NRN GeoPackage",
        path=nrn_path,
        status=nrn_status,
        details=nrn_details,
    )


# =============================================================================
# Validation report
# =============================================================================

print("External silver-input validation")
print("=" * 78)

current_stage: str | None = None

for result in external_input_checks:
    stage_name = str(result["stage"])

    if stage_name != current_stage:
        print(f"\n{stage_name}")
        current_stage = stage_name

    print(
        f"  [{str(result['status']).upper():<7}] "
        f"{result['input']}"
    )
    print(f"            {result['path']}")

    if result["details"] is not None:
        print(f"            {result['details']}")


external_input_failures = [
    result
    for result in external_input_checks
    if result["status"] != "found"
]

stages_with_external_checks = {
    str(result["stage"])
    for result in external_input_checks
}

stages_with_valid_external_inputs = {
    stage_name
    for stage_name in stages_with_external_checks
    if all(
        result["status"] == "found"
        for result in external_input_checks
        if result["stage"] == stage_name
    )
}


print("\nExternal-input summary")
print("=" * 78)
print(f"Input checks:   {len(external_input_checks):,}")
print(
    "Inputs found:  "
    f"{sum(result['status'] == 'found' for result in external_input_checks):,}"
)
print(f"Input failures: {len(external_input_failures):,}")
print(
    "Valid stages:  "
    f"{len(stages_with_valid_external_inputs):,}/"
    f"{len(stages_with_external_checks):,}"
)


if external_input_failures:
    failure_details = "\n".join(
        (
            f"  - {result['stage']} | "
            f"{result['input']} | "
            f"{result['path']}"
        )
        for result in external_input_failures
    )

    raise FileNotFoundError(
        "One or more required external silver-stage inputs are "
        "missing or invalid:\n"
        f"{failure_details}"
    )


print("\nAll required external silver-stage inputs were found.")

External silver-input validation

legacy_inputs
  [FOUND  ] raw provincial boundary shapefile
            C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\raw\basemaps\lpr_000b21a_e.shp

basemaps
  [FOUND  ] raw provincial boundary shapefile
            C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\raw\basemaps\lpr_000b21a_e.shp

legacy_inputs
  [FOUND  ] legacy site table
            C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\sites_full.csv
  [FOUND  ] legacy demand table
            C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\demand.csv

emissions
  [FOUND  ] raw emissions CSV
            C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\raw\emissions\co2_large_facilities_2024\Greenhouse gas emissions from large facilities - 2024.csv
  [FOUND  ] raw emissions GeoJSON
            C:\Users\Andrew Vigars\Research\repos\temoa-u

In [7]:
import subprocess
import sys
from time import perf_counter
from typing import Any


# =============================================================================
# Stage execution
# =============================================================================

completed_silver_stages: set[str] = set()
silver_stage_results: dict[str, Any] = {}


def validate_stage_dependencies(
    stage_name: str,
) -> None:
    """Validate that all upstream silver stages have completed.

    Parameters
    ----------
    stage_name : str
        Name of the silver stage that is about to execute.

    Raises
    ------
    KeyError
        If ``stage_name`` is not registered in the silver workflow.
    RuntimeError
        If one or more required upstream stages have not completed.
    """

    if stage_name not in SILVER_STAGE_DEPENDENCIES:
        raise KeyError(
            f"Unknown silver stage: {stage_name!r}"
        )

    required_stages = SILVER_STAGE_DEPENDENCIES[stage_name]

    incomplete_dependencies = [
        dependency
        for dependency in required_stages
        if dependency not in completed_silver_stages
    ]

    if incomplete_dependencies:
        raise RuntimeError(
            f"Cannot execute {stage_name!r}. "
            "The following upstream stages have not completed: "
            f"{incomplete_dependencies}"
        )


def execute_silver_stage(
    stage_name: str,
    *,
    enforce_dependencies: bool = True,
) -> Any:
    """Execute one registered silver-layer build stage.

    Function-based stages are called directly with the loaded
    ``GeospatialBuildConfig``. Script-based stages are launched in a separate
    Python process from the project root.

    Successful stages are added to ``completed_silver_stages`` and their
    returned values are stored in ``silver_stage_results``. Script-based
    stages return a ``subprocess.CompletedProcess`` object.

    Parameters
    ----------
    stage_name : str
        Registered silver-stage name to execute.
    enforce_dependencies : bool, default=True
        Whether required upstream silver stages must already be marked as
        completed.

    Returns
    -------
    Any
        Value returned by a function-based stage, or the completed subprocess
        record for a script-based stage.

    Raises
    ------
    KeyError
        If ``stage_name`` has no registered executor.
    RuntimeError
        If required upstream stages have not completed.
    ValueError
        If the registered execution method is unsupported.
    subprocess.CalledProcessError
        If a script-based stage exits with a nonzero return code.
    """

    if stage_name not in STAGE_EXECUTORS:
        raise KeyError(
            f"No executor is registered for stage {stage_name!r}."
        )

    if enforce_dependencies:
        validate_stage_dependencies(stage_name)

    executor = STAGE_EXECUTORS[stage_name]
    execution_method = str(executor["method"])

    print("\n" + "=" * 78)
    print(f"Executing silver stage: {stage_name}")
    print(f"Execution method:       {execution_method}")
    print("=" * 78)

    start_time = perf_counter()

    try:
        if execution_method == "function":
            stage_callable = executor["callable"]
            result = stage_callable(build_config)

        elif execution_method == "script":
            script_path = Path(executor["path"])

            result = subprocess.run(
                [
                    sys.executable,
                    str(script_path),
                ],
                cwd=PROJECT_ROOT,
                check=True,
            )

        else:
            raise ValueError(
                f"Unsupported execution method for {stage_name!r}: "
                f"{execution_method!r}"
            )

    except Exception:
        elapsed_seconds = perf_counter() - start_time

        print("\n" + "-" * 78)
        print(f"Stage failed:   {stage_name}")
        print(f"Elapsed time:   {elapsed_seconds:,.2f} seconds")
        print("-" * 78)

        raise

    elapsed_seconds = perf_counter() - start_time

    completed_silver_stages.add(stage_name)
    silver_stage_results[stage_name] = result

    print("\n" + "-" * 78)
    print(f"Stage completed: {stage_name}")
    print(f"Elapsed time:    {elapsed_seconds:,.2f} seconds")
    print("-" * 78)

    return result


print("Silver-stage executor ready.")
print(
    "No processing has been started. "
    "Call execute_silver_stage(stage_name) to run one stage."
)

Silver-stage executor ready.
No processing has been started. Call execute_silver_stage(stage_name) to run one stage.


In [8]:
# =============================================================================
# Full silver workflow execution
# =============================================================================

silver_workflow_history: list[dict[str, object]] = []


def run_silver_workflow(
    *,
    stages: tuple[str, ...] | None = None,
    reset_state: bool = True,
) -> dict[str, Any]:
    """Execute the configured silver-layer workflow in dependency order.

    By default, every stage in ``SILVER_STAGE_ORDER`` is executed. A subset may
    be supplied, but all of its upstream dependencies must either be included
    earlier in the requested stage sequence or already be marked as completed.

    Execution stops immediately when a stage fails. Successfully completed
    stages remain recorded so the failure can be inspected or the workflow can
    be resumed deliberately.

    Parameters
    ----------
    stages : tuple[str, ...] | None, optional
        Ordered silver stages to execute. When ``None``, all registered stages
        are executed using ``SILVER_STAGE_ORDER``.
    reset_state : bool, default=True
        Whether to clear prior completion markers, stage results, and workflow
        history before starting.

    Returns
    -------
    dict[str, Any]
        Stage results keyed by silver-stage name.

    Raises
    ------
    ValueError
        If the requested stage sequence contains unknown stages, duplicates,
        or violates the canonical workflow order.
    RuntimeError
        If a stage cannot run because an upstream dependency is incomplete.
    Exception
        Propagates the original exception raised by the first failed stage.
    """

    selected_stages = (
        tuple(SILVER_STAGE_ORDER)
        if stages is None
        else tuple(stages)
    )

    unknown_stages = [
        stage_name
        for stage_name in selected_stages
        if stage_name not in SILVER_STAGE_ORDER
    ]

    if unknown_stages:
        raise ValueError(
            "The requested workflow contains unknown stages: "
            f"{unknown_stages}"
        )

    if len(selected_stages) != len(set(selected_stages)):
        raise ValueError(
            "The requested workflow contains duplicate stages: "
            f"{selected_stages}"
        )

    selected_positions = [
        SILVER_STAGE_ORDER.index(stage_name)
        for stage_name in selected_stages
    ]

    if selected_positions != sorted(selected_positions):
        raise ValueError(
            "Requested stages must follow SILVER_STAGE_ORDER: "
            f"{selected_stages}"
        )

    if reset_state:
        completed_silver_stages.clear()
        silver_stage_results.clear()
        silver_workflow_history.clear()

    workflow_start_time = perf_counter()

    print("\n" + "=" * 78)
    print("Starting silver-layer workflow")
    print("=" * 78)
    print(f"Build profile: {build_config.source_path}")
    print(f"Study area:    {build_config.study_area.label}")
    print(f"Stage count:   {len(selected_stages):,}")

    for stage_index, stage_name in enumerate(
        selected_stages,
        start=1,
    ):
        stage_start_time = perf_counter()

        print(
            f"\nWorkflow stage {stage_index:,}/"
            f"{len(selected_stages):,}: {stage_name}"
        )

        try:
            result = execute_silver_stage(
                stage_name,
                enforce_dependencies=True,
            )

        except Exception as exc:
            stage_elapsed_seconds = (
                perf_counter() - stage_start_time
            )
            workflow_elapsed_seconds = (
                perf_counter() - workflow_start_time
            )

            silver_workflow_history.append(
                {
                    "stage": stage_name,
                    "status": "failed",
                    "elapsed_seconds": stage_elapsed_seconds,
                    "exception_type": type(exc).__name__,
                    "exception_message": str(exc),
                }
            )

            print("\n" + "=" * 78)
            print("Silver-layer workflow failed")
            print("=" * 78)
            print(f"Failed stage:    {stage_name}")
            print(f"Exception type:  {type(exc).__name__}")
            print(f"Exception:       {exc}")
            print(
                "Workflow elapsed: "
                f"{workflow_elapsed_seconds:,.2f} seconds"
            )
            print(
                "Completed stages:  "
                + (
                    ", ".join(completed_silver_stages)
                    if completed_silver_stages
                    else "none"
                )
            )

            raise

        stage_elapsed_seconds = perf_counter() - stage_start_time

        silver_workflow_history.append(
            {
                "stage": stage_name,
                "status": "completed",
                "elapsed_seconds": stage_elapsed_seconds,
                "exception_type": None,
                "exception_message": None,
            }
        )

        silver_stage_results[stage_name] = result

    workflow_elapsed_seconds = (
        perf_counter() - workflow_start_time
    )

    print("\n" + "=" * 78)
    print("Silver-layer workflow completed")
    print("=" * 78)
    print(
        "Completed stages: "
        f"{len(completed_silver_stages):,}/"
        f"{len(selected_stages):,}"
    )
    print(
        "Total elapsed:    "
        f"{workflow_elapsed_seconds:,.2f} seconds"
    )

    print("\nStage timing summary")
    print("-" * 78)

    for record in silver_workflow_history:
        print(
            f"{str(record['stage']):<28}"
            f"{str(record['status']):<12}"
            f"{float(record['elapsed_seconds']):>12,.2f} seconds"
        )

    return dict(silver_stage_results)


print("Full silver-workflow executor ready.")
print(
    "No stages have been run. "
    "Call run_silver_workflow() to rebuild the silver layer."
)

Full silver-workflow executor ready.
No stages have been run. Call run_silver_workflow() to rebuild the silver layer.


In [9]:
silver_build_results = run_silver_workflow()


Starting silver-layer workflow
Build profile: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\config\provinces_only.toml
Study area:    provinces_only
Stage count:   8

Workflow stage 1/8: legacy_inputs

Executing silver stage: legacy_inputs
Execution method:       function

Boundary source: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\raw\basemaps\lpr_000b21a_e.shp
Configured study area: NL, PE, NS, NB, QC, ON, MB, SK, AB, BC

sites_full: assigning 26,754 point(s) to province polygons...
sites_full: direct polygon join assigned 23,865; 2,889 require nearest province fallback.
sites_full: mapped 26,754; unresolved 0; completed in 52.4 s.

demand: assigning 124 point(s) to province polygons...
demand: direct polygon join assigned 107; 17 require nearest province fallback.
demand: mapped 124; unresolved 0; completed in 3.8 s.

Legacy input mapping complete.
Sites output:  C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geo

0.1deg centroid:   0%|          | 0/181650 [00:00<?, ?cells/s]

Applying provinces_only centroid filter...
Completed geographic grid at 0.1 degree: 80,696 retained cells.
Exported: provinces_only_basemap_0.1deg_centroid.gpkg (80,696 regions)
Exported preview: provinces_only_basemap_0.1deg_centroid.png

Building 0.25° geographic grid using 'centroid' retention...
Grid CRS: EPSG:4326
Native cell size: 0.25 degree
Candidate cells: 29,495


0.25deg centroid:   0%|          | 0/29495 [00:00<?, ?cells/s]

Applying provinces_only centroid filter...
Completed geographic grid at 0.25 degree: 12,913 retained cells.
Exported: provinces_only_basemap_0.25deg_centroid.gpkg (12,913 regions)
Exported preview: provinces_only_basemap_0.25deg_centroid.png

Building 0.5° geographic grid using 'centroid' retention...
Grid CRS: EPSG:4326
Native cell size: 0.5 degree
Candidate cells: 7,482


...

Applying provinces_only centroid filter...
Completed projected grid at 50 km: 2,327 retained cells.
Exported: provinces_only_basemap_50km_centroid.gpkg (2,327 regions)
Exported preview: provinces_only_basemap_50km_centroid.png

Building 75 km projected grid using 'centroid' retention...
Grid CRS: EPSG:3347
Native cell size: 75,000 m (75 km)
Candidate cells: 2,880


75km centroid:   0%|          | 0/2880 [00:00<?, ?cells/s]

Applying provinces_only centroid filter...
Completed projected grid at 75 km: 1,033 retained cells.
Exported: provinces_only_basemap_75km_centroid.gpkg (1,033 regions)
Exported preview: provinces_only_basemap_75km_centroid.png

Building 100 km projected grid using 'centroid' retention...
Grid CRS: EPSG:3347
Native cell size: 100,000 m (100 km)
Candidate cells: 1,650


100km centroid:   0%|          | 0/1650 [00:00<?, ?cells/s]

Applying provinces_only centroid filter...
Completed projected grid at 100 km: 585 retained cells.
Exported: provinces_only_basemap_100km_centroid.gpkg (585 regions)
Exported preview: provinces_only_basemap_100km_centroid.png

Stage 1 complete.
Exported summary: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\basemaps\provinces_only_basemap_summary.csv

------------------------------------------------------------------------------
Stage completed: basemaps
Elapsed time:    259.44 seconds
------------------------------------------------------------------------------

Workflow stage 6/8: adjacency

Executing silver stage: adjacency
Execution method:       function
Found 10 basemap files for profile 'provinces_only':
  - provinces_only_basemap_0.1deg_centroid.gpkg
  - provinces_only_basemap_0.25deg_centroid.gpkg
  - provinces_only_basemap_0.5deg_centroid.gpkg
  - provinces_only_basemap_0.75deg_centroid.gpkg
  - provinces_only_basemap_100km_centroid

In [11]:
# =============================================================================
# Silver build post-run verification
# =============================================================================

expected_completed_stages = set(SILVER_STAGE_ORDER)

missing_completed_stages = sorted(
    expected_completed_stages - completed_silver_stages
)

unexpected_completed_stages = sorted(
    completed_silver_stages - expected_completed_stages
)

failed_stage_records = [
    record
    for record in silver_workflow_history
    if record["status"] != "completed"
]

duplicate_history_stages = sorted(
    {
        record["stage"]
        for record in silver_workflow_history
        if sum(
            other_record["stage"] == record["stage"]
            for other_record in silver_workflow_history
        ) > 1
    }
)

missing_result_entries = sorted(
    expected_completed_stages - set(silver_stage_results)
)


# =============================================================================
# Processed-layer directory checks
# =============================================================================

processed_root = DATA_FILES / "processed"

processed_directory_checks = {
    "processed root": processed_root,
    "legacy inputs": processed_root / "legacy_inputs",
    "emissions": processed_root / "emissions",
    "costs": processed_root / "costs",
    "basemaps": processed_root / "basemaps",
    "graph": processed_root / "graph",
    "road networks": processed_root / "nrn",
    "road connectivity": processed_root / "road_connectivity",
}

missing_processed_directories = {
    label: path
    for label, path in processed_directory_checks.items()
    if not path.is_dir()
}


# =============================================================================
# File inventory
# =============================================================================

processed_files = (
    sorted(
        path
        for path in processed_root.rglob("*")
        if path.is_file()
    )
    if processed_root.is_dir()
    else []
)

files_by_suffix: dict[str, int] = {}

for path in processed_files:
    suffix = path.suffix.lower() or "<no suffix>"
    files_by_suffix[suffix] = files_by_suffix.get(suffix, 0) + 1


# =============================================================================
# Verification report
# =============================================================================

print("Silver build verification")
print("=" * 78)
print(f"Build profile:       {build_config.source_path}")
print(f"Study area:          {build_config.study_area.label}")
print(
    "Completed stages:    "
    f"{len(completed_silver_stages):,}/"
    f"{len(expected_completed_stages):,}"
)
print(f"Workflow records:    {len(silver_workflow_history):,}")
print(f"Stored results:      {len(silver_stage_results):,}")
print(f"Processed files:     {len(processed_files):,}")

print("\nStage status")
print("-" * 78)

for stage_name in SILVER_STAGE_ORDER:
    status = (
        "completed"
        if stage_name in completed_silver_stages
        else "missing"
    )

    print(f"{stage_name:<28}{status}")


print("\nStage timing")
print("-" * 78)

for record in silver_workflow_history:
    print(
        f"{str(record['stage']):<28}"
        f"{str(record['status']):<12}"
        f"{float(record['elapsed_seconds']):>12,.2f} seconds"
    )


print("\nProcessed directories")
print("-" * 78)

for label, path in processed_directory_checks.items():
    status = "found" if path.is_dir() else "missing"
    print(f"{label:<24}{status:<10}{path}")


print("\nProcessed file types")
print("-" * 78)

if files_by_suffix:
    for suffix, count in sorted(files_by_suffix.items()):
        print(f"{suffix:<16}{count:>8,}")
else:
    print("No processed files found.")


# =============================================================================
# Final assertions
# =============================================================================

verification_errors: list[str] = []

if missing_completed_stages:
    verification_errors.append(
        "Missing completed stages: "
        f"{missing_completed_stages}"
    )

if unexpected_completed_stages:
    verification_errors.append(
        "Unexpected completed stages: "
        f"{unexpected_completed_stages}"
    )

if failed_stage_records:
    verification_errors.append(
        "One or more workflow records are not completed: "
        f"{failed_stage_records}"
    )

if duplicate_history_stages:
    verification_errors.append(
        "Duplicate stage records found in workflow history: "
        f"{duplicate_history_stages}"
    )

if missing_result_entries:
    verification_errors.append(
        "Missing stage-result entries: "
        f"{missing_result_entries}"
    )

if missing_processed_directories:
    verification_errors.append(
        "Missing processed directories: "
        + ", ".join(
            f"{label}={path}"
            for label, path in missing_processed_directories.items()
        )
    )

if not processed_files:
    verification_errors.append(
        f"No files were found under {processed_root}"
    )


if verification_errors:
    formatted_errors = "\n".join(
        f"  - {error}"
        for error in verification_errors
    )

    raise RuntimeError(
        "Silver build verification failed:\n"
        f"{formatted_errors}"
    )


print("\nSilver build verification passed.")
print(
    "The configured silver workflow completed and produced a populated "
    "processed layer."
)

Silver build verification
Build profile:       C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\config\provinces_only.toml
Study area:          provinces_only
Completed stages:    8/8
Workflow records:    8
Stored results:      8
Processed files:     192

Stage status
------------------------------------------------------------------------------
legacy_inputs               completed
emissions                   completed
h2_pipeline_costs           completed
h2_pipeline_cost_models     completed
basemaps                    completed
adjacency                   completed
roads                       completed
road_connectivity           completed

Stage timing
------------------------------------------------------------------------------
legacy_inputs               completed          61.39 seconds
emissions                   completed           0.71 seconds
h2_pipeline_costs           completed           0.57 seconds
h2_pipeline_cost_models     completed           0.38 